# 01 Exploración de Datos (EDA)

Este notebook realiza una exploración de datos alineada al reto de **detección de posibles fraudes**.

## Principios
- `ramo` es un campo requerido, pero en este MVP suele ser constante; por eso **no se analiza por ramo**.
- El foco está en señales exigidas por el PDF: **fechas**, **montos**, **documentos**, **proveedores**, **narrativas**.

## Estructura
- Conexión y carga (Supabase)
- Auditoría rápida (nulos, duplicados, tipos)
- Features base (deltas de tiempo y ratios)
- EDA orientado al reto (fraude vs no fraude, cobertura, sucursal, proveedor, documentos)
- Montos atípicos (outliers)
- NLP (PCA de embeddings + narrativas similares con cruce de entidades)


In [1]:
from pathlib import Path
import sys

# Permite importar `src.*` aunque ejecutes desde /notebooks
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

from src.dataio.csv_exports import get_latest_export_dir, load_exported_tables

%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

EXPORT_DIR = get_latest_export_dir(Path("..") / "data" / "raw" / "supabase_export")
print(f"✅ Usando export CSV: {EXPORT_DIR}")


✅ Usando export CSV: ..\data\raw\supabase_export\20260528_094805


In [2]:
tables = load_exported_tables(EXPORT_DIR)

df_asegurados = tables["asegurados"]
df_polizas = tables["polizas"]
df_siniestros = tables["siniestros"]
df_proveedores = tables["proveedores"]
df_documentos = tables["documentos"]

print("✅ Carga completada desde CSV:")
print("- Asegurados:", len(df_asegurados))
print("- Pólizas:", len(df_polizas))
print("- Siniestros:", len(df_siniestros))
print("- Proveedores:", len(df_proveedores))
print("- Documentos:", len(df_documentos))


✅ Carga completada desde CSV:
- Asegurados: 1000
- Pólizas: 1000
- Siniestros: 1000
- Proveedores: 50
- Documentos: 1993


In [3]:
# Auditoría rápida (enfocada en columnas core del reto)
print("--- Auditoría de Siniestros (core) ---")

core_cols = [
    'id_siniestro','id_poliza','id_asegurado','id_proveedor','codigo_siniestro',
    'ramo','cobertura','fecha_ocurrencia','fecha_reporte',
    'monto_reclamado','monto_estimado','monto_pagado',
    'estado','sucursal','descripcion_narrativa','documentos_completos',
    'dias_desde_inicio_poliza','dias_desde_fin_poliza','dias_entre_ocurrencia_reporte',
    'historial_siniestros_asegurado','etiqueta_fraude_simulada'
]
core_cols = [c for c in core_cols if c in df_siniestros.columns]

info = pd.DataFrame({
    'Tipo': df_siniestros[core_cols].dtypes,
    'Nulos': df_siniestros[core_cols].isnull().sum(),
    '% Nulos': (df_siniestros[core_cols].isnull().sum() / max(1, len(df_siniestros))) * 100,
    'Únicos': df_siniestros[core_cols].nunique()
})
display(info)

# Duplicados por código
if 'codigo_siniestro' in df_siniestros.columns:
    dups = df_siniestros.duplicated(subset=['codigo_siniestro']).sum()
    print("Duplicados por codigo_siniestro:", int(dups))

# Conversión fechas
for col in ['fecha_ocurrencia','fecha_reporte']:
    if col in df_siniestros.columns:
        df_siniestros[col] = pd.to_datetime(df_siniestros[col], errors='coerce')

df_polizas['fecha_inicio_vigencia'] = pd.to_datetime(df_polizas.get('fecha_inicio_vigencia'), errors='coerce')
df_polizas['fecha_fin_vigencia'] = pd.to_datetime(df_polizas.get('fecha_fin_vigencia'), errors='coerce')

print("✅ Auditoría completada")


--- Auditoría de Siniestros (core) ---


,Tipo,Nulos,% Nulos,Únicos
id_siniestro,object,0,0.0,1000
id_poliza,object,0,0.0,1000
id_asegurado,object,0,0.0,1000
id_proveedor,object,0,0.0,50
codigo_siniestro,object,0,0.0,1000
ramo,object,0,0.0,1
cobertura,object,0,0.0,3
fecha_ocurrencia,object,0,0.0,748
fecha_reporte,object,0,0.0,750
monto_reclamado,float64,0,0.0,999


Duplicados por codigo_siniestro: 0
✅ Auditoría completada


In [4]:
# Features base (df_master)

df_master = df_siniestros.merge(
    df_polizas[['id_poliza','fecha_inicio_vigencia','monto_asegurado','tipo_cobertura']]
        if 'id_poliza' in df_polizas.columns else df_polizas,
    on='id_poliza',
    how='left'
)

# Deltas (preferir los ya calculados en BD)
if 'delta_vigencia_dias' not in df_master.columns:
    if 'dias_desde_inicio_poliza' in df_master.columns:
        df_master['delta_vigencia_dias'] = df_master['dias_desde_inicio_poliza']
    elif {'fecha_ocurrencia','fecha_inicio_vigencia'}.issubset(df_master.columns):
        df_master['delta_vigencia_dias'] = (df_master['fecha_ocurrencia'] - df_master['fecha_inicio_vigencia']).dt.days

if 'delta_notificacion_dias' not in df_master.columns:
    if 'dias_entre_ocurrencia_reporte' in df_master.columns:
        df_master['delta_notificacion_dias'] = df_master['dias_entre_ocurrencia_reporte']
    elif {'fecha_reporte','fecha_ocurrencia'}.issubset(df_master.columns):
        df_master['delta_notificacion_dias'] = (df_master['fecha_reporte'] - df_master['fecha_ocurrencia']).dt.days

# Transformaciones de monto
if 'monto_reclamado' in df_master.columns:
    df_master['monto_log'] = np.log1p(df_master['monto_reclamado'].astype(float))

if {'monto_reclamado','monto_asegurado'}.issubset(df_master.columns):
    df_master['ratio_reclamado'] = df_master['monto_reclamado'] / df_master['monto_asegurado']

print("✅ df_master listo:", len(df_master))


✅ df_master listo: 1000


In [5]:
def pct(x: pd.Series) -> float:
    return float(x.mean()) if len(x) else np.nan

# Resumen fraude vs no fraude
cols_needed = {
    'etiqueta_fraude_simulada',
    'dias_desde_inicio_poliza',
    'dias_entre_ocurrencia_reporte',
    'documentos_completos',
    'monto_reclamado',
    'cobertura',
}
missing = cols_needed - set(df_master.columns)
if missing:
    raise RuntimeError(f"Faltan columnas para EDA: {missing}")

summary = (
    df_master
    .groupby('etiqueta_fraude_simulada')
    .agg(
        casos=('codigo_siniestro','count'),
        pct_docs_completos=('documentos_completos', pct),
        p50_reporte_tardio=('dias_entre_ocurrencia_reporte', lambda s: s.quantile(0.50)),
        p90_reporte_tardio=('dias_entre_ocurrencia_reporte', lambda s: s.quantile(0.90)),
        p50_borde_inicio=('dias_desde_inicio_poliza', lambda s: s.quantile(0.50)),
        pct_borde_30=('dias_desde_inicio_poliza', lambda s: float((s <= 30).mean())),
        p50_monto=('monto_reclamado', lambda s: s.quantile(0.50)),
        p90_monto=('monto_reclamado', lambda s: s.quantile(0.90)),
    )
)

display(summary)


,casos,pct_docs_completos,p50_reporte_tardio,p90_reporte_tardio,p50_borde_inicio,pct_borde_30,p50_monto,p90_monto
etiqueta_fraude_simulada,,,,,,,,
0,865,0.865896,3.0,5.0,103.0,0.102890,2468.54,6566.844
1,135,0.740741,3.0,5.6,64.0,0.333333,3141.10,9389.088


In [6]:
# Cobertura: volumen y fraude_rate
cov = (
    df_master
    .groupby('cobertura')
    .agg(casos=('codigo_siniestro','count'), fraude_rate=('etiqueta_fraude_simulada','mean'))
    .sort_values('casos', ascending=False)
)

display(cov)

fig = px.bar(
    cov.reset_index(),
    x='cobertura', y='fraude_rate',
    text='casos',
    title='Fraude rate por cobertura (con conteos)'
)
fig.update_yaxes(tickformat='.0%')
fig.show()


,casos,fraude_rate
cobertura,,
ROBO,351,0.128205
CHOQUE,338,0.144970
DAÑOS,311,0.131833


In [7]:
# Sucursal/Ciudad: volumen vs fraude_rate
if 'sucursal' in df_master.columns:
    city = (
        df_master
        .groupby('sucursal')
        .agg(casos=('codigo_siniestro','count'), fraude_rate=('etiqueta_fraude_simulada','mean'))
        .sort_values('casos', ascending=False)
    )
    display(city)

    fig = px.scatter(
        city.reset_index(),
        x='casos', y='fraude_rate',
        size='casos', color='sucursal',
        title='Sucursales: volumen vs fraude rate'
    )
    fig.update_yaxes(tickformat='.0%')
    fig.show()


,casos,fraude_rate
sucursal,,
MANTA,267,0.127341
QUITO,261,0.134100
GUAYAQUIL,243,0.127572
CUENCA,229,0.152838


In [8]:
# Proveedores: concentración y ranking
prov = (
    df_master
    .groupby('id_proveedor')
    .agg(casos=('codigo_siniestro','count'), fraude_rate=('etiqueta_fraude_simulada','mean'))
    .sort_values(['casos','fraude_rate'], ascending=False)
)

display(prov.head(15))

min_casos = max(10, int(len(df_master) * 0.01))
sospechosos = prov[prov['casos'] >= min_casos].sort_values('fraude_rate', ascending=False)
print(f"Top proveedores sospechosos (min_casos={min_casos}):")
display(sospechosos.head(15))


,casos,fraude_rate
id_proveedor,,
prov-7d13740f,67,0.283582
prov-b0d3cc8d,66,0.242424
prov-b6b79676,58,0.224138
prov-0776898a,52,0.192308
prov-7d784f3a,26,0.038462
prov-dcd901e4,24,0.166667
prov-c70782ce,23,0.043478
prov-5184cb01,22,0.045455
prov-cd675ee2,22,0.000000


Top proveedores sospechosos (min_casos=10):


,casos,fraude_rate
id_proveedor,,
prov-7d13740f,67,0.283582
prov-b0d3cc8d,66,0.242424
prov-b6b79676,58,0.224138
prov-013f1a2d,14,0.214286
prov-0ac26afb,15,0.200000
prov-d3fcaebd,20,0.200000
prov-0776898a,52,0.192308
prov-4f575400,11,0.181818
prov-c1720687,17,0.176471


In [9]:
# Documentos: faltantes/inconsistencias por fraude (si están disponibles)
if not df_documentos.empty:
    cols_doc = set(df_documentos.columns)
    base = df_documentos.copy()

    # Indicadores principales del PDF
    for c in ['entregado','es_legible','inconsistencia_detectada','posible_adulteracion']:
        if c not in cols_doc:
            print(f"⚠️ Columna documentos faltante: {c}")

    # Enlazar fraude del siniestro
    if {'id_siniestro'}.issubset(cols_doc) and {'id_siniestro','etiqueta_fraude_simulada'}.issubset(df_siniestros.columns):
        base = base.merge(df_siniestros[['id_siniestro','etiqueta_fraude_simulada']], on='id_siniestro', how='left')

    if 'etiqueta_fraude_simulada' in base.columns:
        agg_cols = [c for c in ['entregado','es_legible','inconsistencia_detectada','posible_adulteracion'] if c in base.columns]
        if agg_cols:
            doc_summary = base.groupby('etiqueta_fraude_simulada')[agg_cols].mean().sort_index()
            print("Promedios de flags en documentos (0=no, 1=sí) por fraude:")
            display(doc_summary)


Promedios de flags en documentos (0=no, 1=sí) por fraude:


,entregado,es_legible,inconsistencia_detectada,posible_adulteracion
etiqueta_fraude_simulada,,,,
0,1.0,0.948807,0.0,0.051193
1,1.0,0.959854,0.0,0.040146


In [10]:
# Montos atípicos por cobertura

plot_df = df_master.copy()
plot_df['fraude'] = plot_df['etiqueta_fraude_simulada'].map({0:'No fraude', 1:'Fraude'})

fig = px.box(
    plot_df,
    x='cobertura', y='monto_reclamado',
    color='fraude',
    points='outliers',
    title='Monto reclamado por cobertura (Fraude vs No fraude)'
)
fig.show()

p99 = df_master.groupby('cobertura')['monto_reclamado'].quantile(0.99)

def es_outlier(row):
    return row['monto_reclamado'] >= p99.loc[row['cobertura']]

outliers = df_master[df_master.apply(es_outlier, axis=1)].copy()
print('Outliers por cobertura (>= p99):', len(outliers))
display(outliers[['codigo_siniestro','cobertura','monto_reclamado','dias_entre_ocurrencia_reporte','documentos_completos','id_proveedor','etiqueta_fraude_simulada']].head(15))


Outliers por cobertura (>= p99): 12


,codigo_siniestro,cobertura,monto_reclamado,dias_entre_ocurrencia_reporte,documentos_completos,id_proveedor,etiqueta_fraude_simulada
147,SIN-2026-0148,ROBO,12390.26,4,True,prov-7d13740f,1
183,SIN-2026-0184,DAÑOS,3270.78,1,True,prov-7d784f3a,1
209,SIN-2026-0210,DAÑOS,3464.04,4,False,prov-0776898a,1
421,SIN-2026-0422,ROBO,11690.67,3,False,prov-4f575400,1
488,SIN-2026-0489,ROBO,12187.67,5,False,prov-f079cfd7,1
622,SIN-2026-0623,ROBO,11046.49,6,False,prov-b0d3cc8d,1
624,SIN-2026-0625,DAÑOS,3487.10,5,True,prov-d3fcaebd,1
704,SIN-2026-0705,CHOQUE,7786.03,4,True,prov-0776898a,1
706,SIN-2026-0707,CHOQUE,8161.99,5,True,prov-0776898a,1
796,SIN-2026-0797,DAÑOS,3785.88,1,True,prov-b0d3cc8d,1


In [11]:
# NLP: Mapa semántico (PCA) + pares similares (cosine)
import ast
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

if 'descripcion_embedding' not in df_master.columns:
    raise RuntimeError('No existe descripcion_embedding en siniestros')

# Embeddings pueden venir como lista o string; normalizamos

def clean_embedding(val):
    if val is None:
        return None
    if isinstance(val, str):
        return np.array(ast.literal_eval(val))
    return np.array(val)

_df_nlp = df_master[df_master['descripcion_embedding'].notnull()].copy()
_df_nlp['emb'] = _df_nlp['descripcion_embedding'].apply(clean_embedding)
_df_nlp = _df_nlp[_df_nlp['emb'].notnull()].copy()
embeddings = np.stack(_df_nlp['emb'].values)

print('✅ Siniestros con embedding:', len(_df_nlp))

# PCA 2D
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)
_df_nlp['pca_x'] = coords[:, 0]
_df_nlp['pca_y'] = coords[:, 1]

fig = px.scatter(
    _df_nlp,
    x='pca_x', y='pca_y',
    color='etiqueta_fraude_simulada',
    hover_data=['codigo_siniestro','cobertura','sucursal'],
    title='Mapa semántico (PCA) de narrativas'
)
fig.show()

# Similitud coseno y pares (filtrado por cruce de entidades)
threshold = 0.95
sim_matrix = cosine_similarity(embeddings)
triu = np.triu(sim_matrix, k=1)
idx = np.where(triu > threshold)

pairs = []
for i, j in zip(*idx):
    s1 = _df_nlp.iloc[i]
    s2 = _df_nlp.iloc[j]

    distinto_asegurado = s1.get('id_asegurado') != s2.get('id_asegurado')
    distinto_proveedor = s1.get('id_proveedor') != s2.get('id_proveedor')
    if not (distinto_asegurado or distinto_proveedor):
        continue

    pairs.append({
        'Siniestro_1': s1['codigo_siniestro'],
        'Siniestro_2': s2['codigo_siniestro'],
        'Similitud': float(triu[i, j]),
        'id_asegurado_1': s1.get('id_asegurado'),
        'id_asegurado_2': s2.get('id_asegurado'),
        'id_proveedor_1': s1.get('id_proveedor'),
        'id_proveedor_2': s2.get('id_proveedor'),
        'Narrativa_1': (s1.get('descripcion_narrativa','')[:140] + '...'),
        'Narrativa_2': (s2.get('descripcion_narrativa','')[:140] + '...'),
    })

pairs_df = pd.DataFrame(pairs)
print(f"🔥 Pares similares (>{threshold}) con cruce de entidades:", len(pairs_df))
if not pairs_df.empty:
    display(pairs_df.sort_values('Similitud', ascending=False).head(10))


✅ Siniestros con embedding: 1000


🔥 Pares similares (>0.95) con cruce de entidades: 329


,Siniestro_1,Siniestro_2,Similitud,id_asegurado_1,id_asegurado_2,id_proveedor_1,id_proveedor_2,Narrativa_1,Narrativa_2
176,SIN-2026-0271,SIN-2026-0356,1.0,ase-74d18a09,ase-55d249a8,prov-9ff54158,prov-ea7fe3bc,EL CLIENTE REPORTA DAÑOS EN PARACHOQUES TRASER...,EL CLIENTE REPORTA DAÑOS EN PARACHOQUES TRASER...
172,SIN-2026-0259,SIN-2026-0322,1.0,ase-f628c1e6,ase-d79dd036,prov-d3fcaebd,prov-5526123f,DAÑO ACCIDENTAL EN LOJA: IMPACTO DE OBJETO EN ...,DAÑO ACCIDENTAL EN LOJA: IMPACTO DE OBJETO EN ...
249,SIN-2026-0479,SIN-2026-0962,1.0,ase-348324dc,ase-ca34722c,prov-1827e2a3,prov-026bdcdd,EL CLIENTE REPORTA DAÑOS EN PARACHOQUES TRASER...,EL CLIENTE REPORTA DAÑOS EN PARACHOQUES TRASER...
279,SIN-2026-0600,SIN-2026-0976,1.0,ase-236a92e1,ase-247629ab,prov-5ddd5177,prov-5526123f,DAÑO ACCIDENTAL EN GUAYAQUIL: IMPACTO DE OBJET...,DAÑO ACCIDENTAL EN GUAYAQUIL: IMPACTO DE OBJET...
43,SIN-2026-0086,SIN-2026-0099,1.0,ase-32721d0a,ase-8fb07356,prov-7d784f3a,prov-5184cb01,DAÑO ACCIDENTAL EN AMBATO: IMPACTO DE OBJETO E...,DAÑO ACCIDENTAL EN AMBATO: IMPACTO DE OBJETO E...
90,SIN-2026-0141,SIN-2026-0694,1.0,ase-0b049f0b,ase-3e164254,prov-dcd901e4,prov-5ddd5177,DAÑO ACCIDENTAL EN LOJA: IMPACTO DE OBJETO EN ...,DAÑO ACCIDENTAL EN LOJA: IMPACTO DE OBJETO EN ...
174,SIN-2026-0263,SIN-2026-0557,1.0,ase-99c289cd,ase-00641e5d,prov-d50cab08,prov-5526123f,DAÑO ACCIDENTAL EN PORTOVIEJO: IMPACTO DE OBJE...,DAÑO ACCIDENTAL EN PORTOVIEJO: IMPACTO DE OBJE...
27,SIN-2026-0057,SIN-2026-0071,1.0,ase-4d262cc0,ase-a3a4dcf2,prov-0776898a,prov-24e38d8f,DAÑO ACCIDENTAL EN CUENCA: IMPACTO DE OBJETO E...,DAÑO ACCIDENTAL EN CUENCA: IMPACTO DE OBJETO E...
97,SIN-2026-0159,SIN-2026-0508,1.0,ase-02b08c42,ase-16cfb6f7,prov-b0d3cc8d,prov-b0d3cc8d,EL CLIENTE REPORTA DAÑOS EN PARABRISAS EN MANT...,EL CLIENTE REPORTA DAÑOS EN PARABRISAS EN MANT...
241,SIN-2026-0446,SIN-2026-0641,1.0,ase-04345502,ase-ec1a714f,prov-1f044dba,prov-7d13740f,DAÑO ACCIDENTAL EN QUITO: IMPACTO DE OBJETO EN...,DAÑO ACCIDENTAL EN QUITO: IMPACTO DE OBJETO EN...
